## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [3]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [4]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


## Para correr con extensión de Colab en Visual Studio Code

In [5]:
import plotly.io as pio

# Set the renderer explicitly for VS Code Notebooks
pio.renderers.default = "vscode"  # alternative: 'notebook_mimetype'


# 1 Exploratory Data Analysis

In [6]:
import os as os
import duckdb
import plotly.express as px

In [7]:
# defino los parametros
PARAM = {'experimento':'EDA-101',
  'kaggle_competition':'labo-iii-2026-ba'
}

In [8]:
# creo la carpeta del experimento
ruta = "/content/buckets/b1/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/EDA-101


## 1.1 Creacion de tablas

Para el Exploratory Data Analysis utilizo DuckDB
<br>Cargo los archivos a utilizar en TABLAS de DuckDB

In [9]:
con = duckdb.connect()

In [10]:
# creo la tabla del sell-in  transformando el campo periodo a tipo DATE

con.execute("""
    CREATE OR REPLACE TABLE tb_sellin AS
    SELECT CAST(strptime(CAST( periodo AS VARCHAR), '%Y%m') AS DATE) periodo,
           customer_id,
           product_id,
           plan_precios_cuidados,
           cust_request_qty,
           cust_request_tn,
           tn
    FROM read_csv_auto('/content/datasets/sell-in.txt.gz')
    ORDER BY product_id, periodo, customer_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
# creo la tabla que posee la definicion de los productos

con.execute("""
    CREATE OR REPLACE TABLE tb_productos AS
    SELECT *
    FROM read_csv_auto('/content/datasets/tb_productos.txt')
    ORDER BY product_id
""")

## 1.2 EDA Ventas Totales

In [12]:
# creo las ventas por producto

con.execute("""
    CREATE OR REPLACE TABLE tb_ventas_global AS
    SELECT periodo,
           SUM(tn) tn
    FROM  tb_sellin
    GROUP BY periodo
    ORDER BY periodo
""")



In [13]:
con.sql("""
SELECT  year(periodo) yean,
        round( SUM(tn) ) tn
FROM    tb_ventas_global
GROUP BY year(periodo)
ORDER BY year(periodo)
""").show()

┌───────┬──────────┐
│ yean  │    tn    │
│ int64 │  double  │
├───────┼──────────┤
│  2017 │ 500310.0 │
│  2018 │ 434448.0 │
│  2019 │ 390230.0 │
└───────┴──────────┘



¿Cómo interpreta lo que viene sucediendo año a año con las ventas?

In [14]:
# Grafico global de ventas
gra = px.line(
    con.sql("SELECT * FROM tb_ventas_global").df(),
    x="periodo",
    y="tn",
    title="Ventas Totales",
    labels={"periodo": "Periodo", "tn": "Toneladas"}
)

# Display the plot
gra.update_yaxes(rangemode="tozero")
gra.show()

In [15]:
# Grafico global de ventas con tendencia
gra = px.scatter(
    con.sql("SELECT * FROM tb_ventas_global").df(),
    x="periodo",
    y="tn",
    title="Ventas Totales con tendencia",
    labels={"periodo": "Periodo", "tn": "Toneladas"},
    trendline='ols',
    trendline_color_override='red'
)

# Display the plot
gra.update_traces(mode='lines')
gra.update_yaxes(rangemode="tozero")
gra.show()

¿Cómo es la tendencia de las ventas en los últimos 3 años?

## 1.3  EDA Clientes
¿Cuánto market share tiene los clientes?

In [16]:
# Creacion de tabla de clientes de 2019
con.execute("""
CREATE OR REPLACE TABLE tb_clientes_2019 AS
SELECT  customer_id,
        SUM(tn) tn,
        (SUM(tn) * 100.0) / SUM(SUM(tn)) OVER () tn_pct
FROM    tb_sellin
WHERE   periodo between '2019-01-01' AND '2019-12-01'
GROUP BY customer_id
ORDER BY tn DESC
""")

In [17]:
gra = px.bar(
    con.sql("SELECT tn_pct FROM tb_clientes_2019").df(),
    y="tn_pct",
    title="Densidad de tn por cliente",
    labels={ "tn_pct": "Toneladas Porcentaje"}
)

# Display the plot
gra.update_yaxes(rangemode="tozero")
gra.show()

In [18]:
# Cantidad de clientes distintos en 2019

con.sql("""
SELECT  COUNT( DISTINCT customer_id )
FROM    tb_clientes_2019
""")

┌─────────────────────────────┐
│ count(DISTINCT customer_id) │
│            int64            │
├─────────────────────────────┤
│                         534 │
└─────────────────────────────┘

In [19]:
# el market share acumulado de los mejores clientes
con.sql("""
SELECT  row_number() OVER(),
        customer_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_clientes_2019
ORDER BY tn DESC
LIMIT 20
""").show()

┌──────────────────────┬─────────────┬────────────────────┬────────────────────┬────────────────────┐
│ row_number() OVER () │ customer_id │         tn         │       tn_pct       │  tn_pct_acumulado  │
│        int64         │    int64    │       double       │       double       │       double       │
├──────────────────────┼─────────────┼────────────────────┼────────────────────┼────────────────────┤
│                    1 │       10001 │ 33685.890009999894 │  8.632321989596592 │  8.632321989596592 │
│                    2 │       10002 │ 25948.003750000018 │ 6.6494168119282735 │ 15.281738801524867 │
│                    3 │       10003 │ 20514.406950000062 │ 5.2570071969435626 │  20.53874599846843 │
│                    4 │       10004 │  15890.07730000008 │  4.071979800814554 │  24.61072579928298 │
│                    5 │       10005 │ 14958.134390000008 │ 3.8331607797747624 │ 28.443886579057743 │
│                    6 │       10006 │ 14147.636789999995 │ 3.6254632466854355 │  

Los primeros 13 productos representan el 50% de las ventas !

In [20]:
# el market share acumulado de los PEORES clientes
con.sql("""
SELECT  row_number() OVER(),
        customer_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_clientes_2019
ORDER BY tn ASC
LIMIT 300
""").show()

┌──────────────────────┬─────────────┬─────────────────────┬────────────────────────┬───────────────────┐
│ row_number() OVER () │ customer_id │         tn          │         tn_pct         │ tn_pct_acumulado  │
│        int64         │    int64    │       double        │         double         │      double       │
├──────────────────────┼─────────────┼─────────────────────┼────────────────────────┼───────────────────┤
│                  534 │       10604 │ 0.19665999999999995 │ 5.0395950410397675e-05 │ 99.99999999999986 │
│                  533 │       10524 │ 0.22125999999999998 │  5.669992874913349e-05 │ 99.99994960404945 │
│                  532 │       10618 │ 0.42128999999999994 │ 0.00010795947294008156 │  99.9998929041207 │
│                  531 │       10574 │ 0.44403000000000004 │ 0.00011378680901418127 │ 99.99978494464776 │
│                  530 │       10581 │             0.50101 │ 0.00012838846290609858 │ 99.99967115783875 │
│                  529 │       10593 │  0.6977

Los 300 clientes menos importantes representan MENOS del 3% de las toneladas vendidas

In [21]:
# evolucion de algunos clientes

clientes=[10001, 10002, 10012]

gra=px.line(
    con.sql(f"SELECT customer_id, periodo, SUM(tn) tn FROM tb_sellin WHERE customer_id in {clientes} GROUP BY customer_id, periodo ORDER BY 1,2").df(),
    x="periodo",
    y="tn",
    color='customer_id',
    title='customer_id =' + str(clientes),
    labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

# Display the plot
gra.update_traces(mode='lines')
gra.update_yaxes(rangemode="tozero")
gra.show()

## 1.4 EDA productos individuales
Market shared de los productos

In [22]:
con.execute("""
CREATE OR REPLACE TABLE tb_productos_201912 AS
SELECT  product_id,
        SUM(tn) tn,
        (SUM(tn) * 100.0) / SUM(SUM(tn)) OVER () tn_pct
FROM    tb_sellin
WHERE   periodo between '2019-01-01' and '2019-12-01'
GROUP BY product_id
ORDER BY tn DESC
""")

In [23]:
con.sql("""
SELECT *
FROM   tb_productos_201912
""").show()

┌────────────┬──────────────────────┬────────────────────────┐
│ product_id │          tn          │         tn_pct         │
│   int64    │        double        │         double         │
├────────────┼──────────────────────┼────────────────────────┤
│      20001 │    17456.79264000003 │      4.473465149039151 │
│      20002 │   14105.245700000058 │     3.6146001363962186 │
│      20003 │    9419.716889999925 │      2.413889887462744 │
│      20005 │    8019.241249999978 │     2.0550050054104325 │
│      20004 │    7526.583939999965 │     1.9287569968470204 │
│      20009 │    6495.871040000001 │     1.6646272490805425 │
│      20032 │    6493.670259999988 │     1.6640632787777618 │
│      20006 │    5743.364500000031 │     1.4717904633928656 │
│      20007 │     5209.65367000002 │      1.335022109268118 │
│      20010 │    5154.895930000037 │     1.3209899301283614 │
│        ·   │            ·         │              ·         │
│        ·   │            ·         │              ·   

In [24]:
con.sql("""
SELECT  product_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_productos_201912
ORDER BY tn DESC
""").show()

┌────────────┬──────────────────────┬────────────────────────┬────────────────────┐
│ product_id │          tn          │         tn_pct         │  tn_pct_acumulado  │
│   int64    │        double        │         double         │       double       │
├────────────┼──────────────────────┼────────────────────────┼────────────────────┤
│      20001 │    17456.79264000003 │      4.473465149039151 │  4.473465149039151 │
│      20002 │   14105.245700000058 │     3.6146001363962186 │   8.08806528543537 │
│      20003 │    9419.716889999925 │      2.413889887462744 │ 10.501955172898114 │
│      20005 │    8019.241249999978 │     2.0550050054104325 │ 12.556960178308547 │
│      20004 │    7526.583939999965 │     1.9287569968470204 │ 14.485717175155568 │
│      20009 │    6495.871040000001 │     1.6646272490805425 │  16.15034442423611 │
│      20032 │    6493.670259999988 │     1.6640632787777618 │  17.81440770301387 │
│      20006 │    5743.364500000031 │     1.4717904633928656 │ 19.2861981664

In [25]:
def graficar_un_producto(producto):
  gra = px.scatter(
      con.sql(f"SELECT periodo, SUM(tn) tn FROM tb_sellin WHERE product_id={producto} GROUP BY periodo").df(),
      x="periodo",
      y="tn",
      title=con.sql(f"SELECT CONCAT_WS(', ', *COLUMNS('.*')) FROM tb_productos WHERE product_id={producto}").fetchone()[0],
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_traces(mode='lines')
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [26]:
def graficar_productos(productos):
    for prod in productos:
      graficar_un_producto(prod)

In [27]:
# gnero graficos INDEPENDIENTES de productos

graficar_productos( [20001, 20002, 20003, 20004])

## 1.5  EDA multiples productos

In [28]:
productos = [ 20001, 20002]

tbl=con.sql(f"SELECT product_id, periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY product_id, periodo ORDER BY 1, 2").df()
display(tbl)

,product_id,periodo,tn
0,20001,2017-01-01,934.77222
1,20001,2017-02-01,798.01620
2,20001,2017-03-01,1303.35771
3,20001,2017-04-01,1069.96130
4,20001,2017-05-01,1502.20132
...,...,...,...
67,20002,2019-08-01,813.78215
68,20002,2019-09-01,1090.18771
69,20002,2019-10-01,1979.53635
70,20002,2019-11-01,1423.57739


In [29]:
def graficar_multiples_productos(productos):
  gra = px.line(
      con.sql(f"SELECT product_id, periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY product_id, periodo ORDER BY 1,2").df(),
      x="periodo",
      y="tn",
      color="product_id",
      title="Multiples Productos",
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [30]:
def graficar_union_productos(productos):
  gra = px.line(
      con.sql(f"SELECT periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY periodo ORDER BY 1").df(),
      x="periodo",
      y="tn",
      title="Productos in " + str(productos),
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [31]:
graficar_multiples_productos( [20001, 20002, 20003] )

In [32]:
graficar_union_productos( [20001, 20002, 20003] )

## 1.6  EDA Estacionalidad Mayonesas

In [33]:
con.sql("SELECT * FROM tb_productos WHERE cat3='Mayonesa'")

┌─────────┬──────────┬──────────┬─────────┬──────────┬────────────┬──────────────────────┐
│  cat1   │   cat2   │   cat3   │  brand  │ sku_size │ product_id │     descripcion      │
│ varchar │ varchar  │ varchar  │ varchar │  int64   │   int64    │       varchar        │
├─────────┼──────────┼──────────┼─────────┼──────────┼────────────┼──────────────────────┤
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      475 │      20003 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      240 │      20004 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      120 │      20005 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      950 │      20019 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      475 │      20046 │ Light sin TACC       │
│ FOODS   │ ADEREZOS │ Mayonesa │ MAYOS3  │      475 │      20084 │ Reguar sin TACC      │
│ FOODS   │ ADEREZOS │ Mayonesa │ MAJESTA │      475 │      20107 │ Mayonesa Tradicional │

In [34]:
algunas_mayonesas = [20003, 20004, 20005]

In [35]:
con.sql(f"SELECT * FROM tb_productos WHERE product_id in {productos}")

┌─────────┬─────────────┬─────────┬─────────┬──────────┬────────────┬────────────────────┐
│  cat1   │    cat2     │  cat3   │  brand  │ sku_size │ product_id │    descripcion     │
│ varchar │   varchar   │ varchar │ varchar │  int64   │   int64    │      varchar       │
├─────────┼─────────────┼─────────┼─────────┼──────────┼────────────┼────────────────────┤
│ HC      │ ROPA LAVADO │ Liquido │ ARIEL   │     3000 │      20001 │ genoma             │
│ HC      │ ROPA LAVADO │ Liquido │ LIMPIEX │     3000 │      20002 │ Maquina 1er lavado │
└─────────┴─────────────┴─────────┴─────────┴──────────┴────────────┴────────────────────┘

In [36]:
graficar_multiples_productos(algunas_mayonesas)

In [37]:
graficar_union_productos(algunas_mayonesas)

¿Qué estacionalidad se observa para las mayonesas?

## 1.7 EDA Estacionalidad Sopas
Dado que entre que usualmente hay DOS MESES entre que el caminón sale por el portón de la planta de La Multinacional y es escaneado en la linea de caja del supermercado
<br> ¿En qué mes las familias empiezan a compar mas sopas?
<br> ¿En qué mes la multinacional vende más sopas?

In [38]:
con.sql("SELECT * FROM tb_productos WHERE cat3='Sopas'")

┌─────────┬────────────────┬─────────┬─────────┬──────────┬────────────┬─────────────────────────┐
│  cat1   │      cat2      │  cat3   │  brand  │ sku_size │ product_id │       descripcion       │
│ varchar │    varchar     │ varchar │ varchar │  int64   │   int64    │         varchar         │
├─────────┼────────────────┼─────────┼─────────┼──────────┼────────────┼─────────────────────────┤
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20234 │ Sopa 21                 │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20265 │ Sopa vegetales          │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20302 │ Sopa fideos y vegetales │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20305 │ Sopa fideos             │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20350 │ Sopla 22                │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20398 │ Sopa 1                  │
│ FOODS   

In [39]:
algunas_sopas=[20234, 20265, 20302]

In [40]:
graficar_multiples_productos(algunas_sopas)

In [41]:
graficar_union_productos(algunas_sopas)

## 1.8 EDA Productos Infantiles
¿Qué forma tiene el sell-out de los productos nuevos?
<br>¿Los productos nuevos, canibalizan a otros productos de la misma familia de productos de La Multinacional?

In [42]:
mostazas_infantes=[21144, 21146, 21154]

In [43]:
graficar_multiples_productos(mostazas_infantes)

In [44]:
mostazas_varias=[21144, 21146, 21154, 20884]

In [45]:
con.sql(f"SELECT * FROM tb_productos WHERE product_id in {mostazas_varias}")

┌─────────┬──────────┬─────────┬──────────┬──────────┬────────────┬──────────────────────────┐
│  cat1   │   cat2   │  cat3   │  brand   │ sku_size │ product_id │       descripcion        │
│ varchar │ varchar  │ varchar │ varchar  │  int64   │   int64    │         varchar          │
├─────────┼──────────┼─────────┼──────────┼──────────┼────────────┼──────────────────────────┤
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      275 │      20884 │ abejas                   │
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      180 │      21144 │ Pimienta                 │
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      180 │      21146 │ colchon de finas hierbas │
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      180 │      21154 │ Mostaza Ahumada          │
└─────────┴──────────┴─────────┴──────────┴──────────┴────────────┴──────────────────────────┘

In [46]:
graficar_multiples_productos(mostazas_varias)

Podria pasar que para 2019-09 el product_id=20884 se vendió menos porque fue canibalizado por los nuevos tres productos?

## 1.9 EDA Productos discontinuados

¿Los productos discontinuados son reemplazados por otros que toman su lugar?

In [47]:
mayonesas_discontinuadas=[20494, 20554]

In [48]:
graficar_multiples_productos(mayonesas_discontinuadas)

In [49]:
mayonesas=[20494, 20554, 20580]

In [50]:
graficar_multiples_productos(mayonesas)